# Tunisian ASR — Final Dataset Builder (`03_dataset_builder`)

**Input:**
- `data/interim/audio_clean/{train,test}/segments.arrow` + `manifest.parquet`
- `data/interim/text_clean/df_{train,test}_clean.parquet`

**Output:** `data/processed/final/` — HuggingFace `DatasetDict` ready for Whisper / XLSR fine-tuning.

### Road-map
| § | Section | Action |
|---|---------|--------|
| 1 | Environment & config | — |
| 2 | Load Phase 2 outputs | Arrow waveforms + manifest |
| 3 | Load Phase 3 outputs | text-clean parquets |
| 4 | Inspect & validate raw join | column audit, null check |
| 5 | Filter & clean | kept==True, drop intermediates |
| 6 | Build HuggingFace dataset | cast Audio feature |
| 7 | Final validation | schema, sample, stats |
| 8 | Save | DatasetDict + dataset card |

## 1 · Environment & Config

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent.resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

from src.utils           import load_config, ensure_dir, print_dict, print_section, df_overview
from src.dataset_builder import (
    load_arrow_waveforms, build_audio_column,
    drop_unnecessary_columns, validate_dataset,
    compute_final_stats, write_dataset_card,
    FINAL_COLUMNS, _MANIFEST_PRIORITY_COLS,
)

print("✓ imports OK")

✓ imports OK


In [5]:
paths = load_config(f"{PROJECT_ROOT}/configs/paths.yaml")

AUDIO_CLEAN_DIR = Path(PROJECT_ROOT) / paths["interim"]["audio_clean"]
TEXT_CLEAN_DIR  = Path(PROJECT_ROOT) / paths["interim"]["text_clean"]
OUTPUT_DIR      = Path(PROJECT_ROOT) / paths["processed"]["final"]

ensure_dir(OUTPUT_DIR)

print(f"AUDIO_CLEAN_DIR : {AUDIO_CLEAN_DIR}")
print(f"TEXT_CLEAN_DIR  : {TEXT_CLEAN_DIR}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")

AUDIO_CLEAN_DIR : C:\Users\MSI\Desktop\ASR\asr_tn_pielines\data\interim\audio_clean
TEXT_CLEAN_DIR  : C:\Users\MSI\Desktop\ASR\asr_tn_pielines\data\interim\text_clean
OUTPUT_DIR      : C:\Users\MSI\Desktop\ASR\asr_tn_pielines\data\processed\final


## 2 · Load Phase 2 Outputs

Load the Arrow waveform file and the full manifest (which includes all segments,
kept and dropped, with quality metrics).

In [7]:
manifest_train = pd.read_parquet(AUDIO_CLEAN_DIR / "train" / "manifest.parquet")
manifest_test  = pd.read_parquet(AUDIO_CLEAN_DIR / "test"  / "manifest.parquet")

print(f"Manifest train : {manifest_train.shape[0]:,} rows × {manifest_train.shape[1]} cols")
print(f"Manifest test  : {manifest_test.shape[0]:,} rows  × {manifest_test.shape[1]} cols")
print("\nColumns:", manifest_train.columns.tolist())

Manifest train : 76,080 rows × 29 cols
Manifest test  : 1,013 rows  × 29 cols

Columns: ['transcript', 'transcript_raw', 'char_count', 'word_count', 'has_latin', 'has_diacritics', 'speech_rate', 'transcript_clean', 'transcript_cs', 'transcript_nota', 'alef_applied', 'taa_applied', 'waw_applied', 'hamza_applied', 'variant_applied', 'override_applied', 'negation_applied', 'audio_id', 'split', 'seg_start', 'seg_end', 'sample_rate', 'duration_s', 'rms', 'peak', 'silence_ratio', 'kept', 'drop_reason', 'smoothing_applied']


In [8]:
# Quality gate breakdown — how many were kept/dropped and why
for label, mdf in [("train", manifest_train), ("test", manifest_test)]:
    n_kept    = mdf["kept"].sum()
    n_dropped = (~mdf["kept"]).sum()
    reasons   = mdf[~mdf["kept"]]["drop_reason"].value_counts().to_dict()
    print(f"{label}: kept={n_kept:,}  dropped={n_dropped:,}  reasons={reasons}")

train: kept=72,314  dropped=3,766  reasons={'too_short': 3588, 'mostly_silent': 161, 'too_long': 14, 'too_silent': 3}
test: kept=1,005  dropped=8  reasons={'too_short': 7, 'mostly_silent': 1}


In [9]:
import time

t0 = time.perf_counter()
# Load Arrow waveforms (kept segments only, as written by save_segments_arrow)
waves_train = load_arrow_waveforms(AUDIO_CLEAN_DIR / "train" / "segments.arrow")
waves_test  = load_arrow_waveforms(AUDIO_CLEAN_DIR / "test"  / "segments.arrow")
elapsed = time.perf_counter() - t0
print(f"✓ Time elapsed: {elapsed/60:.1f} min")

# print(f"Arrow train : {len(waves_train):,} waveforms")
print(f"Arrow test  : {len(waves_test):,}  waveforms")
# print(f"Columns     : {waves_train.columns.tolist()}")

# Spot-check one waveform
w = waves_test.iloc[1]
print(w)
print(f"\nSample — audio_id={w['audio_id']}")
print(f"  waveform dtype : {w['waveform'].dtype}  shape: {w['waveform'].shape}")
print(f"  sample_rate    : {w['sample_rate']}")
print(f"  duration_s     : {w['duration_s']:.3f}")

from IPython.display import Audio

# Play it
Audio(data=w["waveform"], rate=w["sample_rate"])

✓ Time elapsed: 1.7 min
Arrow test  : 1,013  waveforms
audio_id            amenyKH_0000002_chunk_0000076_002_chunk_76_0
waveform       [0.0050354004, 0.00033569336, -0.0015563965, -...
sample_rate                                                16000
duration_s                                                  2.84
Name: 1, dtype: object

Sample — audio_id=amenyKH_0000002_chunk_0000076_002_chunk_76_0
  waveform dtype : float32  shape: (45440,)
  sample_rate    : 16000
  duration_s     : 2.840


## 3 · Load Phase 3 Outputs

We need the `transcript_nota` column — the final NOTA-normalised label.

In [11]:
text_train = pd.read_parquet(TEXT_CLEAN_DIR / "df_train_clean.parquet")
text_test  = pd.read_parquet(TEXT_CLEAN_DIR / "df_test_clean.parquet")

print(f"Text-clean train : {text_train.shape[0]:,} rows × {text_train.shape[1]} cols")
print(f"Text-clean test  : {text_test.shape[0]:,} rows  × {text_test.shape[1]} cols")
print("\nColumns:", text_train.columns.tolist())

Text-clean train : 76,080 rows × 21 cols
Text-clean test  : 1,013 rows  × 21 cols

Columns: ['audio_id', 'seg_start', 'seg_end', 'transcript', 'transcript_raw', 'char_count', 'word_count', 'has_latin', 'has_diacritics', 'seg_duration', 'speech_rate', 'transcript_clean', 'transcript_cs', 'transcript_nota', 'alef_applied', 'taa_applied', 'waw_applied', 'hamza_applied', 'variant_applied', 'override_applied', 'negation_applied']


In [12]:
# We only need audio_id + transcript_nota from the text side
text_train = text_train[["audio_id", "transcript_nota"]].copy()
text_test  = text_test[["audio_id",  "transcript_nota"]].copy()

# Basic transcript quality check
for label, tdf in [("train", text_train), ("test", text_test)]:
    n_null  = tdf["transcript_nota"].isna().sum()
    n_empty = (tdf["transcript_nota"].str.strip().str.len() == 0).sum()
    print(f"{label}: null={n_null}  empty={n_empty}  "
          f"sample='{tdf['transcript_nota'].iloc[0]}'")

train: null=0  empty=0  sample='اسبقية قبل انا ما وصلت خممت فيه كيما باش نحكيو من بعد الا ما انا كانطروبرونور كباعث مشروع صارولي برشا مشاكل فالجستين و صارولي مشاكل مع لعباد لي كانت موفرتلي اللوجسيل ولا اللوجسيل اوف لنيه ولا لوجسيل بيراتي'
test: null=0  empty=0  sample='ويرحمني و اياكم في الدنيا و الآخرة'


## 4 · Inspect & Validate Raw Join

In [14]:
# ── Kept-only manifest subset ─────────────────────────────────────────────
kept_train = manifest_train[manifest_train["kept"] == True].copy()
kept_test  = manifest_test[manifest_test["kept"]  == True].copy()

print(f"Kept train : {len(kept_train):,}  Kept test : {len(kept_test):,}")

# drop duplicate columns
kept_train = kept_train.drop(columns=_MANIFEST_PRIORITY_COLS)
kept_test = kept_test.drop(columns=_MANIFEST_PRIORITY_COLS)

# ── Three-way join: manifest (kept) + waveforms + transcript ──────────────
df_train = (kept_train
            .merge(waves_train, on="audio_id", how="inner")
            .merge(text_train,  on="audio_id", how="inner"))

df_test  = (kept_test
            .merge(waves_test,  on="audio_id", how="inner")
            .merge(text_test,   on="audio_id", how="inner"))

print(f"\nAfter join — train: {len(df_train):,}  test: {len(df_test):,}")
print(f"Columns ({len(df_train.columns)}): {df_train.columns.tolist()}")

Kept train : 72,314  Kept test : 1,005

After join — train: 72,314  test: 1,005
Columns (30): ['transcript', 'transcript_raw', 'char_count', 'word_count', 'has_latin', 'has_diacritics', 'speech_rate', 'transcript_clean', 'transcript_cs', 'alef_applied', 'taa_applied', 'waw_applied', 'hamza_applied', 'variant_applied', 'override_applied', 'negation_applied', 'audio_id', 'split', 'seg_start', 'seg_end', 'rms', 'peak', 'silence_ratio', 'kept', 'drop_reason', 'smoothing_applied', 'waveform', 'sample_rate', 'duration_s', 'transcript_nota']


In [15]:
import gc
del waves_train, waves_test
gc.collect()

34

In [16]:
# ── Null audit ────────────────────────────────────────────────────────────
print("Null counts (train):")
null_counts = df_train.isnull().sum()
print(null_counts[null_counts > 0].to_string() or "  none")

print("\nNull counts (test):")
null_counts_te = df_test.isnull().sum()
print(null_counts_te[null_counts_te > 0].to_string() or "  none")

Null counts (train):
Series([], )

Null counts (test):
Series([], )


In [17]:
# ── Sample rows ───────────────────────────────────────────────────────────
df_train[["audio_id","seg_start","seg_end","duration_s",
          "rms","transcript_nota"]].head(5)

,audio_id,seg_start,seg_end,duration_s,rms,transcript_nota
0,amenyKH_0000001_0000020_1_20_0,0.0,14.113000,12.976,0.059691,اسبقية قبل انا ما وصلت خممت فيه كيما باش نحكيو...
1,amenyKH_0000001_chunk_0000001_001_chunk_1_0,0.0,3.496625,2.416,0.085900,و الصلاة و السلام على رسول الله
2,amenyKH_0000001_chunk_0000002_001_chunk_2_0,0.0,4.328063,3.344,0.076495,و على آله وصحابته و من والاه
3,amenyKH_0000001_chunk_0000003_001_chunk_3_0,0.0,4.297500,3.232,0.069927,ربنا لا تزغ قلوبنا بعد اذ هديتنا
4,amenyKH_0000001_chunk_0000004_001_chunk_4_0,0.0,3.285187,2.256,0.073522,وهب لنا من لدنك رحمة


## 5 · Filter & Clean

- Rename `transcript_nota` → `text` (the model-facing label column).
- Build the HuggingFace `audio` dict column from waveform + sample_rate.
- Drop all columns that are not needed for training.

In [19]:
# Rename label column
df_train = df_train.rename(columns={"transcript_nota": "text"})
df_test  = df_test.rename(columns={"transcript_nota": "text"})

# Build audio dict column ({"array": ..., "sampling_rate": ...})
df_train["audio"] = df_train.apply(build_audio_column, axis=1)
df_test["audio"]  = df_test.apply(build_audio_column, axis=1)

print("audio column sample:", type(df_train["audio"].iloc[0]),
      list(df_train["audio"].iloc[0].keys()))

audio column sample: <class 'dict'> ['array', 'sampling_rate']


In [20]:
# Drop intermediate and analysis columns
df_train_final = drop_unnecessary_columns(df_train)
df_test_final  = drop_unnecessary_columns(df_test)

print(f"Before drop: train {df_train.shape[1]} cols  test {df_test.shape[1]} cols")
print(f"After  drop: train {df_train_final.shape[1]} cols  test {df_test_final.shape[1]} cols")
print(f"\nFinal columns: {df_train_final.columns.tolist()}")

Before drop: train 31 cols  test 31 cols
After  drop: train 7 cols  test 7 cols

Final columns: ['audio_id', 'split', 'seg_start', 'seg_end', 'duration_s', 'text', 'audio']


In [21]:
del df_train
del df_test
gc.collect()

0

In [22]:
# Null audit on final DataFrame
for label, df in [("train", df_train_final), ("test", df_test_final)]:
    nulls = df.isnull().sum()
    bad = nulls[nulls > 0]
    print(f"{label}: {'✓ no nulls' if len(bad) == 0 else bad.to_string()}")

train: ✓ no nulls
test: ✓ no nulls


## 6 · Build HuggingFace Dataset

In [ ]:
from datasets import Dataset, DatasetDict, Audio as HFAudio

# Convert to HuggingFace Dataset and cast the audio column
hf_train = Dataset.from_pandas(df_train_final, preserve_index=False)
hf_train = hf_train.cast_column("audio", HFAudio(sampling_rate=16000))

hf_test  = Dataset.from_pandas(df_test_final, preserve_index=False)
hf_test  = hf_test.cast_column("audio", HFAudio(sampling_rate=16000))

dataset_dict = DatasetDict({"train": hf_train, "test": hf_test})

print(dataset_dict)
print("\nFeatures:")
for name, feat in dataset_dict["train"].features.items():
    print(f"  {name:<20} {feat}")

In [ ]:
del df_train_final, df_test_final
gc.collect()

## 7 · Final Validation

In [ ]:
# Run validation checks
all_warnings = []
for split_name, df in [("train", df_train_final), ("test", df_test_final)]:
    warnings = validate_dataset(df, split_name)
    all_warnings.extend(warnings)
    status = "✓ passed" if not warnings else f"⚠ {len(warnings)} warnings"
    print(f"{split_name}: {status}")
    for w in warnings:
        print(f"  {w}")

if not all_warnings:
    print("\n✓ All validation checks passed.")

In [ ]:
# Compute and display final statistics
stats = [compute_final_stats(df_train_final, "train"),
         compute_final_stats(df_test_final,  "test")]

comparison = pd.DataFrame([{
    "split":     s["split"],
    "segments":  s["n_segments"],
    "hours":     s["total_hours"],
    "mean_dur":  s["mean_duration_s"],
    "median_dur": s["median_duration_s"],
    "min_dur":   s["min_duration_s"],
    "max_dur":   s["max_duration_s"],
} for s in stats])
print(comparison.to_string(index=False))

In [ ]:
# Decode one sample to confirm end-to-end round-trip
sample = hf_train[0]
print(f"audio_id   : {sample['audio_id']}")
print(f"transcript : {sample['transcript']}")
print(f"duration_s : {sample['duration_s']:.3f}")
print(f"audio.keys : {list(sample['audio'].keys())}")
print(f"audio.array: shape={sample['audio']['array'].shape}  "
      f"sr={sample['audio']['sampling_rate']}  "
      f"dtype={sample['audio']['array'].dtype}")

## 8 · Save

In [ ]:
dataset_dict.save_to_disk(str(OUTPUT_DIR))
write_dataset_card(stats, OUTPUT_DIR)

print(f"✓ DatasetDict saved → {OUTPUT_DIR}")
print(f"\nTo reload:")
print(f"  from datasets import load_from_disk")
print(f"  ds = load_from_disk('{OUTPUT_DIR}')") 

In [ ]:
print_section("DATASET BUILDER — SUMMARY")
for s in stats:
    print_dict({
        "Segments   ": f"{s['n_segments']:,}",
        "Total hours": f"{s['total_hours']:.3f} h",
        "Mean dur   ": f"{s['mean_duration_s']:.3f} s",
        "Columns    ": str(s['columns']),
    }, title=f"Split: {s['split']}")